# KonfAI Synthesis Demo

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vboussot/KonfAI/blob/main/examples/Synthesis/Synthesis_demo.ipynb)

This notebook is meant to work from a **fresh environment**, including **Google Colab**.

It will help you:

- bootstrap KonfAI if the runtime is empty
- download the public `Synthesis` demo subset
- inspect one case
- prepare the standard `TRAIN -> PREDICTION -> EVALUATION` workflow
- optionally launch the commands step by step

If you are on Colab, enabling a GPU runtime is recommended before training.
        


In [ ]:
from pathlib import Path
import importlib.util
import os
import subprocess
import sys


def find_repo_root(start: Path):
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "examples").exists():
            return candidate
    return None


IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_DIR = Path("/content/KonfAI")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "https://github.com/vboussot/KonfAI", str(REPO_DIR)], check=True)
else:
    REPO_DIR = find_repo_root(Path.cwd().resolve())
    if REPO_DIR is None:
        raise RuntimeError(
            "Could not locate the KonfAI repository. Run this notebook from the repo or open it in Google Colab."
        )

EXAMPLE_DIR = REPO_DIR / "examples" / "Synthesis"
DATASET_DIR = EXAMPLE_DIR / "Dataset"
print("Repository:", REPO_DIR)
print("Example:", EXAMPLE_DIR)
print("Dataset:", DATASET_DIR)

required_packages = {
    "konfai": str(REPO_DIR),
    "SimpleITK": "SimpleITK",
    "huggingface_hub": "huggingface_hub",
    "matplotlib": "matplotlib",
}
missing_packages = [package for module, package in required_packages.items() if importlib.util.find_spec(module) is None]

if missing_packages:
    print("Installing missing dependencies:", ", ".join(missing_packages))
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing_packages], check=True)
else:
    print("KonfAI environment already available.")


## 1. Download the public synthesis demo subset

The demo data lives on Hugging Face in `VBoussot/konfai-demo`.

The cell below uses the `hf` CLI to download only the `Synthesis` subset, then removes the local Hugging Face metadata cache created under `Dataset/.cache`.
        


In [ ]:
import shutil

RUN_DOWNLOAD = True

if RUN_DOWNLOAD:
    subprocess.run([
        "hf",
        "download",
        "VBoussot/konfai-demo",
        "--repo-type",
        "dataset",
        "--include",
        "Synthesis/**",
        "--local-dir",
        str(DATASET_DIR),
    ], check=True)

nested_dir = DATASET_DIR / "Synthesis"
if nested_dir.exists():
    for item in nested_dir.iterdir():
        target = DATASET_DIR / item.name
        if target.exists():
            if target.is_dir():
                shutil.rmtree(target)
            else:
                target.unlink()
        shutil.move(str(item), str(target))
    shutil.rmtree(nested_dir)

shutil.rmtree(DATASET_DIR / ".cache", ignore_errors=True)

cases = sorted([path for path in DATASET_DIR.iterdir() if path.is_dir()]) if DATASET_DIR.exists() else []
print("Dataset ready:", DATASET_DIR.exists())
print("Number of cases:", len(cases))
if cases:
    print("First case:", cases[0].name)
        


## 2. Inspect one case

This quick sanity check reads one sample, prints basic information for the `MR`, `CT`, and `MASK` groups, and displays a representative slice.


In [ ]:
import matplotlib.pyplot as plt
import SimpleITK as sitk
import numpy as np


def describe_image(path: Path):
    image = sitk.ReadImage(str(path))
    array = sitk.GetArrayFromImage(image)
    return image, array, {
        "shape": tuple(array.shape),
        "dtype": str(array.dtype),
        "min": float(np.min(array)),
        "max": float(np.max(array)),
        "spacing": tuple(float(x) for x in image.GetSpacing()),
    }

cases = sorted([path for path in DATASET_DIR.iterdir() if path.is_dir()]) if DATASET_DIR.exists() else []
if not cases:
    raise RuntimeError("Dataset is empty. Run the download cell above first.")

case = cases[0]
print("Sample case:", case.name)
loaded = {}
for group in ["MR", "CT", "MASK"]:
    _, array, stats = describe_image(case / f"{group}.mha")
    loaded[group] = array
    print(group, stats)

mask_array = loaded["MASK"]
if mask_array.ndim >= 3:
    scores = mask_array.reshape(mask_array.shape[0], -1).sum(axis=1)
    slice_index = int(np.argmax(scores)) if np.any(scores) else int(mask_array.shape[0] // 2)
else:
    slice_index = 0

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, group, cmap in zip(axes, ["MR", "CT", "MASK"], ["gray", "gray", "viridis"]):
    image = loaded[group]
    slice_2d = image[slice_index] if image.ndim >= 3 else image
    ax.imshow(slice_2d, cmap=cmap)
    ax.set_title(f"{group} - slice {slice_index}")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 3. Review the workflow files and commands

This example now includes two synthesis variants:

- the baseline workflow with `Config.yml`
- an optional GAN workflow with `Config_GAN.yml`
- shared `Prediction.yml`, `Evaluation.yml`, and `Model.py` files reused by both variants

The baseline is the easiest first run.
The GAN variant demonstrates a 2D / 2.5D generator, a 3D discriminator, and the `;accu;` patch-accumulation semantic used by KonfAI.


In [ ]:
import tempfile
import torch


def latest_checkpoint(train_name: str):
    checkpoint_dir = EXAMPLE_DIR / "Checkpoints" / train_name
    checkpoints = sorted(checkpoint_dir.glob("*.pt"), key=lambda path: path.stat().st_mtime) if checkpoint_dir.exists() else []
    return checkpoints[-1] if checkpoints else None

BASELINE_FILES = ["Config.yml", "Prediction.yml", "Evaluation.yml", "Model.py", "UnNormalize.py"]
GAN_FILES = ["Config_GAN.yml", "Prediction.yml", "Evaluation.yml", "Model.py", "UnNormalize.py"]


RUNTIME_DIR = Path(tempfile.mkdtemp(prefix="konfai-synthesis-"))


def prepare_runtime_configs(train_name: str):
    prediction_runtime = RUNTIME_DIR / f"Prediction_{train_name}.yml"
    evaluation_runtime = RUNTIME_DIR / f"Evaluation_{train_name}.yml"

    prediction_text = (EXAMPLE_DIR / "Prediction.yml").read_text(encoding="utf-8")
    prediction_text = prediction_text.replace("train_name: TRAIN_01", f"train_name: {train_name}")
    prediction_runtime.write_text(prediction_text, encoding="utf-8")

    evaluation_text = (EXAMPLE_DIR / "Evaluation.yml").read_text(encoding="utf-8")
    evaluation_text = evaluation_text.replace("./Predictions/TRAIN_01/Dataset:i:mha", f"./Predictions/{train_name}/Dataset:i:mha")
    evaluation_text = evaluation_text.replace("train_name: TRAIN_01", f"train_name: {train_name}")
    evaluation_runtime.write_text(evaluation_text, encoding="utf-8")

    return prediction_runtime, evaluation_runtime

print("Baseline files:")
for name in BASELINE_FILES:
    print(" -", name, "->", EXAMPLE_DIR / name)

print("\nGAN files:")
for name in GAN_FILES:
    print(" -", name, "->", EXAMPLE_DIR / name)

device_args = ["--gpu", "0"] if torch.cuda.is_available() else ["--cpu", "1"]

baseline_checkpoint = latest_checkpoint("TRAIN_01")
gan_checkpoint = latest_checkpoint("TRAIN_GAN_01")

baseline_prediction_config, baseline_evaluation_config = prepare_runtime_configs("TRAIN_01")
gan_prediction_config, gan_evaluation_config = prepare_runtime_configs("TRAIN_GAN_01")

BASELINE_COMMANDS = {
    "TRAIN": ["konfai", "TRAIN", "-y", *device_args, "--config", "Config.yml"],
    "PREDICTION": [
        "konfai",
        "PREDICTION",
        "-y",
        *device_args,
        "--config",
        baseline_prediction_config.name,
        "--models",
        str(baseline_checkpoint.relative_to(EXAMPLE_DIR)) if baseline_checkpoint else "Checkpoints/TRAIN_01/<checkpoint>.pt",
    ],
    "EVALUATION": ["konfai", "EVALUATION", "-y", "--config", baseline_evaluation_config.name],
}

GAN_COMMANDS = {
    "TRAIN": ["konfai", "TRAIN", "-y", *device_args, "--config", "Config_GAN.yml"],
    "PREDICTION": [
        "konfai",
        "PREDICTION",
        "-y",
        *device_args,
        "--config",
        gan_prediction_config.name,
        "--models",
        str(gan_checkpoint.relative_to(EXAMPLE_DIR)) if gan_checkpoint else "Checkpoints/TRAIN_GAN_01/<checkpoint>.pt",
    ],
    "EVALUATION": ["konfai", "EVALUATION", "-y", "--config", gan_evaluation_config.name],
}

print("\nBaseline commands:")
for name, cmd in BASELINE_COMMANDS.items():
    print(name, " ".join(cmd))

print("\nGAN commands:")
for name, cmd in GAN_COMMANDS.items():
    print(name, " ".join(cmd))


## 4. Optionally run the workflow

Keep all flags set to `False` if you only want to inspect the setup.

Set `USE_GAN_VARIANT = True` if you want to switch from the baseline synthesis workflow to the GAN workflow.
When prediction or evaluation is enabled, the notebook automatically creates temporary runtime configs from the shared `Prediction.yml` and `Evaluation.yml` files, with the right `train_name` for the selected experiment.


In [ ]:
USE_GAN_VARIANT = False
RUN_TRAIN = False
RUN_PREDICTION = False
RUN_EVALUATION = False

COMMANDS = GAN_COMMANDS if USE_GAN_VARIANT else BASELINE_COMMANDS
TRAIN_NAME = "TRAIN_GAN_01" if USE_GAN_VARIANT else "TRAIN_01"


def run_command(cmd):
    print("$", " ".join(cmd))
    subprocess.run(cmd, check=True, cwd=EXAMPLE_DIR)

if RUN_TRAIN:
    run_command(COMMANDS["TRAIN"])

if RUN_PREDICTION:
    checkpoint = latest_checkpoint(TRAIN_NAME)
    if checkpoint is None:
        raise RuntimeError(f"No checkpoint found in Checkpoints/{TRAIN_NAME}. Run training first.")
    run_command(COMMANDS["PREDICTION"][:-1] + [str(checkpoint.relative_to(EXAMPLE_DIR))])

if RUN_EVALUATION:
    run_command(COMMANDS["EVALUATION"])


## 5. Expected outputs

After a complete run, you should find outputs under the selected `TRAIN_NAME`, for example:

- `Checkpoints/TRAIN_01/` or `Checkpoints/TRAIN_GAN_01/`
- `Statistics/TRAIN_01/` or `Statistics/TRAIN_GAN_01/`
- `Predictions/TRAIN_01/` or `Predictions/TRAIN_GAN_01/`
- `Evaluations/TRAIN_01/` or `Evaluations/TRAIN_GAN_01/`

Once that works, the next step is usually to adapt the YAML files to your own paired dataset and model choices.
        
